In [2]:
import re
import pandas as pd
import glob
import os
import numpy as np

In [3]:
folder = "../data/raw/liqudated"

In [4]:
# import pandas as pd
# import glob
# import os

# folder = "../data/liqudated"

# dfs = []
# for file in glob.glob(os.path.join(folder, "*.xlsx")):
#     df = pd.read_excel(file, skiprows=3)
#     dfs.append(df)

# merged = pd.concat(dfs, ignore_index=True, join="outer")
# merged.to_csv("../data/processed/bancrupted/merged_bancruts.csv", index=False)

In [5]:
merged = pd.read_csv('../data/processed/bancrupted/merged_bancruts.csv')

/var/folders/rl/0ykyj_7s1wx56ylgms0h9bxh0000gn/T/ipykernel_32695/33887739.py:1: DtypeWarning: Columns (74,99,100,113) have mixed types. Specify dtype option on import or set low_memory=False.
  merged = pd.read_csv('../data/processed/bancrupted/merged_bancruts.csv')


In [6]:
merged = merged[merged['Важная информация'].str.contains('банкрот', case=False, na=False)]

In [7]:
merged = merged.drop(columns=[c for c in merged.columns if 'Среднесписочная численность' in str(c)],
                     errors='ignore')

meta_cols = ['Регистрационный номер', 'Возраст компании, лет',
             'Дата ликвидации', 'Вид деятельности/отрасль', 'Форма собственности', 'Важная информация']

for c in meta_cols:
    if c not in merged.columns:
        merged[c] = pd.NA

id_cols = ['№', 'Наименование'] + meta_cols

value_cols = [c for c in merged.columns if re.match(r'\d{4},', str(c))]

df_long = merged.melt(id_vars=id_cols, value_vars=value_cols,
                      var_name='Показатель_год', value_name='Значение')
df_long[['Год', 'Показатель']] = df_long['Показатель_год'].str.extract(r'(\d{4}),\s*(.*)')
df_long = df_long.drop(columns='Показатель_год')

df_long['Год'] = pd.to_numeric(df_long['Год'], errors='coerce').astype('Int64')

pivot_index = ['Наименование', 'Год'] + meta_cols
df_wide = df_long.pivot_table(
    index=pivot_index,
    columns='Показатель',
    values='Значение',
    aggfunc='first'
).reset_index()
df_wide.columns.name = None

In [8]:
df_wide = df_wide.drop(columns='Коэффициент покрытия процентов по EBIT, %')

df_wide['Дата ликвидации'] = pd.to_datetime(df_wide['Дата ликвидации']).apply(lambda x: x.year)
df_wide = df_wide.sort_values(['Дата ликвидации', 'Регистрационный номер', 'Год'])

counts = df_wide['Регистрационный номер'].value_counts().sort_values()
df_wide = df_wide[df_wide['Регистрационный номер'].isin(counts[counts > 2].index)]

In [9]:
df = df_wide.dropna(axis=0, subset=df_wide.loc[:,'Коэффициент быстрой ликвидности, %':'Соотношение совокупного долга к капиталу, %'].columns.to_list(), thresh=6)

In [10]:
df.loc[:,'next_year_liqudation'] = df.apply(lambda x: x['Дата ликвидации'] - x['Год'] if (x['Дата ликвидации'] - x['Год'] == 1) else 0, axis=1)

/var/folders/rl/0ykyj_7s1wx56ylgms0h9bxh0000gn/T/ipykernel_32695/251777823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,'next_year_liqudation'] = df.apply(lambda x: x['Дата ликвидации'] - x['Год'] if (x['Дата ликвидации'] - x['Год'] == 1) else 0, axis=1)


In [11]:
has_next_year_liqudation = df.groupby('Регистрационный номер')['next_year_liqudation'].sum()
df =df[df['Регистрационный номер'].isin(has_next_year_liqudation[has_next_year_liqudation>=1].index)]

In [12]:
counts = df['Регистрационный номер'].value_counts().sort_values()
df = df[df['Регистрационный номер'].isin(counts[counts > 2].index)]

In [13]:
year_of_group = df.groupby('Регистрационный номер')['Год']
is_same_size = (year_of_group.last() - year_of_group.first() == year_of_group.count()-1)
df = df[df['Регистрационный номер'].isin(is_same_size[is_same_size].index)]


In [14]:
# df.to_csv('bankrupted.csv')

In [15]:
companies_ages = df.groupby('Регистрационный номер')['Возраст компании, лет'].first().sort_values()
print(f'Средний возраст: {companies_ages.mean():.1f}, Макс: {companies_ages.max()}, Мин: {companies_ages.min()}, медиана: {companies_ages.median()}')

Средний возраст: 18.9, Макс: 36.0, Мин: 4.0, медиана: 18.0


In [16]:
companies_revenue = df.groupby('Регистрационный номер')['Выручка, RUB'].mean().sort_values()
print(f'Средняя выручка: {companies_revenue.mean():,.1f}, Макс: {companies_revenue.max():,.1f}, Мин: {companies_revenue.min():,.1f}, медиана: {companies_revenue.median():,.1f}')

Средняя выручка: 161,553,165.1, Макс: 3,426,754,200.0, Мин: 31,500.0, медиана: 52,220,000.0


In [17]:
df.groupby('Вид деятельности/отрасль')['Регистрационный номер'].nunique().sort_values(ascending=False)

Вид деятельности/отрасль
Строительство жилых и нежилых зданий                                                                 89
Управление эксплуатацией жилого фонда за вознаграждение или на договорной основе                     42
Выращивание зерновых культур                                                                         21
Торговля оптовая неспециализированная                                                                16
Аренда и управление собственным или арендованным нежилым недвижимым имуществом                       16
                                                                                                     ..
Производство машин и оборудования общего назначения                                                   1
Производство машин и оборудования для производства пищевых продуктов, напитков и табачных изделий     1
Производство машин и оборудования для металлургии                                                     1
Производство кузнечно-прессового оборуд

In [18]:
for col in ['Оборачиваемость кредиторской задолженности, разы', 'Оборачиваемость дебиторской задолженности, разы']:
    df.loc[:, col] = df[col].apply(lambda x: np.nan if x <= 0 else x)
    df.loc[:, col] = df.groupby('Вид деятельности/отрасль')[col].transform(
    lambda x: x.fillna(x.median()))

In [22]:
for col in ['Рентабельность продаж, %', 'Соотношение совокупного долга к капиталу, %', 'Выручка, RUB']:
    df.loc[:,col] = df.groupby('Регистрационный номер')[col].transform(
    lambda x: x.fillna(x.mean()))

In [20]:
for column in df.loc[:, 'Выручка, RUB':'Соотношение совокупного долга к капиталу, %'].columns.to_list():
    low, high = df.loc[:, column].quantile([0.01, 0.99])
    df.loc[:, column] = df.loc[:, column].clip(lower=low, upper=high)

In [ ]:
df.to_csv('../data/processed/bancrupted/bancrupted_clean.csv')